In [ ]:

# IMPORT LIBRARY

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

print("Semua library berhasil di-import!")


In [ ]:

# Melihat dimensi awal dataset
print("Dimensi awal dataset:", df.shape)

# Melihat informasi setiap kolom
df.info()

# Menampilkan 5 baris pertama
display(df.head())


In [ ]:

# Menghitung jumlah data duplikat
jumlah_duplikat = df.duplicated().sum()

print("Jumlah data duplikat:", jumlah_duplikat)

# Menampilkan sampel data duplikat
display(df[df.duplicated(keep=False)].head(10))

# Menghapus seluruh data duplikat
df = df.drop_duplicates()

# Memeriksa kembali jumlah data
print("Dimensi setelah menghapus duplikat:", df.shape)


In [ ]:

print("Sisa baris data:", len(df))


In [ ]:

# Melihat kategori sebelum dibersihkan
print("Sebelum dibersihkan:")
print(df['Status_Pernikahan'].unique())

# Membersihkan spasi dan huruf kapital
df['Status_Pernikahan'] = (
    df['Status_Pernikahan']
    .astype('string')
    .str.strip()
    .str.title()
)

# Melihat kategori setelah dibersihkan
print("\nSetelah dibersihkan:")
print(df['Status_Pernikahan'].unique())


In [ ]:

# Mengubah kolom Usia menjadi numerik
df['Usia'] = pd.to_numeric(df['Usia'], errors='coerce')

# Mengubah usia di luar batas 17-100 menjadi NaN
df.loc[
    (df['Usia'] < 17) | (df['Usia'] > 100),
    'Usia'
] = np.nan

print("Jumlah usia yang tidak valid:")
print(df['Usia'].isna().sum())


In [ ]:
# Load the external dataset first to define df_eksternal
try:
    df_eksternal = pd.read_csv('data_kredit_eksternal.csv')
except FileNotFoundError:
    # In case the file is not found, print a descriptive message
    print("Error: 'data_kredit_eksternal.csv' not found in workspace. Please upload the file.")
    df_eksternal = None

if df_eksternal is not None:
    # Menggabungkan dataset utama dengan dataset eksternal
    df = pd.merge(
        df,
        df_eksternal,
        on='ID_Pelanggan',
        how='left',
        suffixes=('', '_eksternal')
    )

    # Melihat hasil penggabungan
    display(df.head())

    # Melihat dimensi setelah penggabungan
    print("Dimensi setelah merge:", df.shape)

In [ ]:

print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])


In [ ]:

# Menghitung persentase missing value
missing_percentage = (
    df.isnull().sum() / len(df) * 100
)

# Membuat tabel missing value
missing_table = pd.DataFrame({
    'Jumlah Missing': df.isnull().sum(),
    'Persentase Missing (%)': missing_percentage
})

print("Persentase Missing Values:")
display(missing_table)


In [ ]:

# Kolom numerik kontinu menggunakan median
kolom_numerik = [
    'Usia',
    'Pendapatan_Bulanan',
    'Lama_Bekerja_Thn'
]

for kolom in kolom_numerik:
    df[kolom] = pd.to_numeric(df[kolom], errors='coerce')
    df[kolom] = df[kolom].fillna(df[kolom].median())


# Kolom diskrit menggunakan nilai 0
df['Jumlah_Kartu_Kredit'] = (
    pd.to_numeric(df['Jumlah_Kartu_Kredit'], errors='coerce')
    .fillna(0)
)


# Kolom kategorikal menggunakan modus
kolom_kategorikal = [
    'Tingkat_Pendidikan',
    'Pernah_Gagal_Bayar_Lain'
]

for kolom in kolom_kategorikal:
    modus = df[kolom].mode(dropna=True)

    if not modus.empty:
        df[kolom] = df[kolom].fillna(modus.iloc[0])


print("Missing value berhasil ditangani.")


In [ ]:

# Memeriksa missing value setelah pengisian
print("Total missing value:",
      df.isnull().sum().sum())

# Menampilkan missing value setiap kolom
display(df.isnull().sum())


In [ ]:

plt.figure(figsize=(8, 5))

sns.boxplot(
    x=df['Pendapatan_Bulanan']
)

plt.title('Box Plot Pendapatan Bulanan')
plt.xlabel('Pendapatan Bulanan')

plt.show()


In [ ]:

# Menghitung kuartil pertama dan ketiga
Q1 = df['Pendapatan_Bulanan'].quantile(0.25)
Q3 = df['Pendapatan_Bulanan'].quantile(0.75)

# Menghitung Interquartile Range
IQR = Q3 - Q1

# Menghitung batas bawah dan batas atas
batas_bawah = Q1 - (1.5 * IQR)
batas_atas = Q3 + (1.5 * IQR)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Batas bawah:", batas_bawah)
print("Batas atas:", batas_atas)


In [ ]:

# Menghitung jumlah outlier sebelum penanganan
jumlah_outlier = (
    (df['Pendapatan_Bulanan'] < batas_bawah) |
    (df['Pendapatan_Bulanan'] > batas_atas)
).sum()

print("Jumlah outlier sebelum penanganan:",
      jumlah_outlier)

# Melakukan Capping (Winsorization)
df['Pendapatan_Bulanan'] = df[
    'Pendapatan_Bulanan'
].clip(
    lower=batas_bawah,
    upper=batas_atas
)

# Visualisasi setelah penanganan
plt.figure(figsize=(8, 5))

sns.boxplot(x=df['Pendapatan_Bulanan'])

plt.title('Box Plot Setelah Penanganan Outlier')
plt.xlabel('Pendapatan Bulanan')

plt.show()


In [ ]:

# Menyeragamkan teks pendidikan
df['Tingkat_Pendidikan'] = (
    df['Tingkat_Pendidikan']
    .astype(str)
    .str.strip()
    .str.title()
)

# Mengubah kategori pendidikan menjadi angka
mapping_pendidikan = {
    'Sma': 1,
    'Sarjana': 2,
    'Magister': 3
}

df['Tingkat_Pendidikan'] = (
    df['Tingkat_Pendidikan']
    .map(mapping_pendidikan)
)

print(df['Tingkat_Pendidikan'].value_counts(dropna=False))


In [ ]:

# Membersihkan format teks
df['Pernah_Gagal_Bayar_Lain'] = (
    df['Pernah_Gagal_Bayar_Lain']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Mengubah kategori menjadi angka biner
mapping_biner = {
    'ya': 1,
    'tidak': 0,
    'yes': 1,
    'no': 0,
    '1': 1,
    '0': 0
}

df['Pernah_Gagal_Bayar_Lain'] = (
    df['Pernah_Gagal_Bayar_Lain']
    .map(mapping_biner)
)

print(df['Pernah_Gagal_Bayar_Lain'].value_counts(dropna=False))


In [ ]:

# Melakukan One-Hot Encoding
df = pd.get_dummies(
    df,
    columns=['Status_Pernikahan'],
    drop_first=True,
    dtype=int
)

# Melihat hasil encoding
display(df.head())


In [ ]:

# Menghapus kolom identifier
df = df.drop(
    columns=['ID_Pelanggan'],
    errors='ignore'
)

# Melihat tipe data kolom
print(df.dtypes)


In [ ]:

# Memisahkan fitur prediktor dan target
X = df.drop(columns=['Status_Default'])
y = df['Status_Default']

print("Dimensi X:", X.shape)
print("Dimensi y:", y.shape)

# Memastikan kolom target tidak ikut menjadi prediktor
display(X.head())


In [ ]:

# Memilih seluruh fitur numerik
fitur_numerik = X.select_dtypes(
    include=['int64', 'float64']
).columns

# Menyimpan statistik sebelum standarisasi
data_sebelum = X[fitur_numerik].describe()

# Membuat objek StandardScaler
scaler = StandardScaler()

# Melakukan standarisasi
X[fitur_numerik] = scaler.fit_transform(
    X[fitur_numerik]
)

print("Standarisasi berhasil dilakukan.")


In [ ]:

# Statistik sebelum standarisasi
print("STATISTIK SEBELUM STANDARISASI")
display(data_sebelum[['Usia',
                      'Pendapatan_Bulanan',
                      'Lama_Bekerja_Thn']])

# Statistik sesudah standarisasi
print("STATISTIK SESUDAH STANDARISASI")
display(
    X[['Usia',
       'Pendapatan_Bulanan',
       'Lama_Bekerja_Thn']].describe()
)


In [ ]:

# Menggabungkan kembali fitur dan target
df_final = pd.concat([X, y], axis=1)

# Menyimpan dataset hasil preprocessing
df_final.to_csv(
    'clean_nasabah_final.csv',
    index=False
)

print("Dataset berhasil disimpan!")
print("Nama file: clean_nasabah_final.csv")

# Memeriksa hasil akhir
print("Dimensi dataset akhir:", df_final.shape)

display(df_final.head())


In [ ]:
print("Total missing values in df_final:", df_final.isnull().sum().sum())

# Detail missing values per column
missing_info = pd.DataFrame({
    'Missing Count': df_final.isnull().sum(),
    'Percentage (%)': (df_final.isnull().sum() / len(df_final)) * 100
})

display(missing_info)

In [ ]:
# SIMPAN HASIL AKHIR

In [ ]:

from google.colab import files

files.download('clean_nasabah_final.csv')


A.* Pemeriksaan Awal dan Penghapusan Duplikat*

Kesimpulan: Dataset awal memiliki 508 baris dan 9 kolom. Setelah dilakukan penghapusan data duplikat sebanyak 8 baris, tersisa 500 baris data.

B. *Penanganan Kesalahan Data*

Kesimpulan: Data status pernikahan berhasil diseragamkan dan data usia yang tidak sesuai rentang 17–100 tahun diubah menjadi nilai kosong agar tidak memengaruhi hasil analisis.

C. *Integrasi Data*

Kesimpulan: Penggabungan data dilakukan menggunakan metode Left Join berdasarkan ID_Pelanggan. Namun, berdasarkan hasil output, file data_kredit_eksternal.csv belum berhasil ditemukan sehingga proses penggabungan belum berjalan dengan sempurna.

D. *Penanganan Missing Values*

Kesimpulan: Data kosong ditangani menggunakan median, modus, dan nilai 0 sesuai jenis kolomnya agar dataset menjadi lebih lengkap.

E.* Penanganan Outlier*

Kesimpulan: Outlier pada kolom Pendapatan_Bulanan diperiksa menggunakan Box Plot dan metode IQR, kemudian ditangani dengan teknik Capping tanpa menghapus baris data.

F. *Pengkodean Data Kategorikal*

Kesimpulan: Data kategorikal berhasil diubah menjadi bentuk numerik menggunakan Ordinal Encoding, Binary Mapping, dan One-Hot Encoding agar dapat digunakan dalam proses Machine Learning.

G. *Feature Scaling*

Kesimpulan: Fitur prediktor dan target dipisahkan, kemudian fitur numerik distandarisasi menggunakan StandardScaler agar memiliki skala yang lebih seragam.
Kesimpulan Akhir
Proses Data Preprocessing dilakukan untuk membersihkan, memperbaiki, dan mempersiapkan data agar lebih terstruktur sebelum digunakan dalam pemodelan Machine Learning.

JADI YAG TERPENTING ADALAH:

Di bagian C masih ada kendala karena file data_kredit_eksternal.csv tidak ditemukan. Jadi, proses integrasi data belum berhasil sepenuhnya. Sebaiknya unggah ulang file tersebut dan jalankan kembali bagian C sampai G agar hasil akhirnya benar-benar sesuai instruksi tugas dosen.